In [1]:
import ray
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from ray import tune
from ray.tune import Tuner, TuneConfig
from ray import serve
import torch
import requests
from requests import Request
import pandas as pd
import json
from intro import clean_and_combine, VectorizeAndEncode, ProductClassifier, train_loop, train_driver

# Introduction - Ray - Anyscale - AI Libraries - Ray Core

## What is Ray?

<img src='https://docs.ray.io/en/releases-2.38.0/_images/map-of-ray.svg' width=700 />

## Ray

* OSS framework for high-performance, resilient, scale-out computation on heterogeneous hardware
* Distributed scheduler supporting stateless functions ("tasks") as well as long-running stateful processes ("actors")
* Key features: 
  * Dependency tracking (task graphs)
  * Data movement and resource aware
  * Supports mix of resource requirements (e.g., GPUs), fractional, and custom resources
* Additional infra: object store, fault tolerance via GCS
* Ray AI Libraries are a set of high-level APIs for accomplishing common large-scale data + compute use cases (e.g., data transformation, model training)
* Easy, Python-based APIs and coding patterns

## Anyscale: Production-ready Ray from day one

* __Developer central__: multi-node backed IDE, advanced observability by Ray library
* __Optimized runtime__: faster performance and higher GPU utilization vs. OSS
* __Cluster controller__: proactive unhealthy node replacement, 0-100 node 60-sec cold starts
* __Expertise__: Training, 24/7 support, professional services


# Overview of the Ray AI Libraries

Built on top of Ray Core, the Ray AI Libraries inherit all the performance and scalability benefits offered by Core while providing a convenient abstraction layer for machine learning. These Python-first native libraries allow ML practitioners to distribute individual workloads, end-to-end applications, and build custom use cases in a unified framework.

The Ray AI Libraries bring together an ever-growing ecosystem of integrations with popular machine learning frameworks to create a common interface for development.

|<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Introduction_to_Ray_AIR/e2e_air.png" width="100%" loading="lazy">|
|:-:|
|Ray AI Libraries enable end-to-end ML development and provides multiple options for integrating with other tools and libraries form the MLOps ecosystem.|



# End-to-End Demo: Product Category Prediction 

## Load and process data with Ray Data

In [2]:
ds = ray.data.read_csv("s3://anyscale-public-materials-use2/ecom/intro/ecommerce_product_catalog.csv")

pd.DataFrame(ds.take(5))

2026-08-24 09:39:52,187	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 100.87.9.27:6379...
2026-08-24 09:39:52,218	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-gy6zbz8z924rvbk9ur6wsep7e1.i.anyscaleuserdata.com 
2026-08-24 09:39:52,221	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_b54fc61a3fd3be5ea6847e78f3bc6cc33520278b.zip' (0.32MiB) to Ray cluster...
2026-08-24 09:39:52,223	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_b54fc61a3fd3be5ea6847e78f3bc6cc33520278b.zip'.
/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-08-24 09:39:54,329	INFO dataset.py:3818 -- Tip: Use `take_batch()` i

,product_id,title,description,category,brand,price,rating,num_reviews
0,PROD-0052,TechPro Ultra Gaming Mouse,Sleek design with durable build quality. High-...,Electronics,TechPro,68.83,3.5,1827
1,PROD-0574,IronPulse All-Terrain Jump Rope,Take your training to the next level. Great fo...,Sports & Outdoors,IronPulse,94.61,3.4,1013
2,PROD-0099,TechPro Advanced Phone Mount,Ideal for home office or gaming setups. Compat...,Electronics,TechPro,53.94,3.0,514
3,PROD-0273,UrbanThread Vintage Cardigan,Available in a range of sizes for the perfect ...,Clothing,UrbanThread,81.91,4.7,1735
4,PROD-0366,BrightSpace Professional Silicone Spatula Set,"Perfect for meal prep, cooking, and entertaini...",Home & Kitchen,BrightSpace,57.17,2.9,751


(autoscaler +1m42s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


### Clean and combine text fields

We lowercase, strip punctuation, and concatenate `title` and `description` into a single `text` column. This is a **per-row** transform, so we use `ds.map()`.

In [3]:
ds_cleaned = ds.map(clean_and_combine)

pd.DataFrame(ds_cleaned.take(5))

2026-08-24 09:43:52,961	INFO logging.py:416 -- Registered dataset logger for dataset dataset_3_0
2026-08-24 09:43:52,966	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_3_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 09:43:52,966	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_3_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> LimitOperator[limit=5] -> TaskPoolMapOperator[Map(clean_and_combine)]
2026-08-24 09:43:52,995	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_3_0 =======
2026-08-24 09:43:52,996	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 09:43:52,997	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 09:43:52,998	INFO logging_progress.py:181 -- 
2026-08-24 09:43:52,998	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 09:43:52,999	INFO loggin

,product_id,title,description,category,brand,price,rating,num_reviews,text
0,PROD-0052,TechPro Ultra Gaming Mouse,Sleek design with durable build quality. High-...,Electronics,TechPro,68.83,3.5,1827,techpro ultra gaming mouse sleek design with d...
1,PROD-0574,IronPulse All-Terrain Jump Rope,Take your training to the next level. Great fo...,Sports & Outdoors,IronPulse,94.61,3.4,1013,ironpulse allterrain jump rope take your train...
2,PROD-0099,TechPro Advanced Phone Mount,Ideal for home office or gaming setups. Compat...,Electronics,TechPro,53.94,3.0,514,techpro advanced phone mount ideal for home of...
3,PROD-0273,UrbanThread Vintage Cardigan,Available in a range of sizes for the perfect ...,Clothing,UrbanThread,81.91,4.7,1735,urbanthread vintage cardigan available in a ra...
4,PROD-0366,BrightSpace Professional Silicone Spatula Set,"Perfect for meal prep, cooking, and entertaini...",Home & Kitchen,BrightSpace,57.17,2.9,751,brightspace professional silicone spatula set ...


### Vectorize and encode with `map_batches`

Now we apply the fitted vectorizer and the label mapping in a single `map_batches` call. This is a **stateful, batch transform**. Ray Data will run this in parallel across its block partitions.

In [4]:
ds_encoded = ds_cleaned.map_batches(VectorizeAndEncode)

pd.DataFrame(ds_encoded.take(5))

2026-08-24 09:44:11,484	INFO logging.py:416 -- Registered dataset logger for dataset dataset_5_0
2026-08-24 09:44:11,491	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_5_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 09:44:11,492	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Map(clean_and_combine)->MapBatches(VectorizeAndEncode)] -> LimitOperator[limit=5]
{"asctime":"2026-08-24 09:44:11,521","levelname":"E","message":"Actor with class name: 'MapWorker(Map(clean_and_combine)->MapBatches(VectorizeAndEncode))' and ID: 'bf732e4eee098637d26185cd03000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more detail

,features,label
0,"[0.0, 0.0, 0.0, 0.0, 0.23569702, 0.0, 0.086035...",2
1,"[0.0, 0.0, 0.0, 0.38758785, 0.0, 0.0, 0.164480...",4
2,"[0.0, 0.0, 0.29161227, 0.0, 0.31964785, 0.0, 0...",2
3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.08521276, 0.0...",1
4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.12161146, 0.0...",3


### Train / test split

In [5]:
train_ds, val_ds = ds_encoded.train_test_split(test_size=0.2, seed=42)

print(f"Train: {train_ds.count()} rows")
print(f"Val:   {val_ds.count()} rows")

2026-08-24 09:44:36,502	INFO logging.py:416 -- Registered dataset logger for dataset dataset_6_0
2026-08-24 09:44:36,507	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_6_0. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
2026-08-24 09:44:36,508	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_6_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[Map(clean_and_combine)->MapBatches(VectorizeAndEncode)]
2026-08-24 09:44:36,653	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_6_0 =======
2026-08-24 09:44:36,654	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 09:44:36,655	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store (pending: 1 CPU)
2026-08-24 09:44:36,656	INFO logging_progress.py:181 -- 
2026-08-24 09:44:36,656	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 

Train: 800 rows
Val:   200 rows


## Train a PyTorch classifier with Ray Train

We define a small feedforward network and train it using `TorchTrainer`.

In [6]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop,
    scaling_config=ScalingConfig(num_workers=2),
    datasets={"train": train_ds},
    run_config=RunConfig(storage_path='/mnt/cluster_storage/'),
    train_loop_config={"lr" : 1e-2}
)

result = trainer.fit()

result.metrics_dataframe

(TrainController pid=62733) Requesting resources: {'CPU': 1} * 2
(TrainController pid=62733) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=62733) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(RayTrainWorker pid=42620, ip=100.105.27.67) [Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=42619, ip=100.105.27.67) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=62733) Started training worker group of size 2: 
(TrainController pid=62733) - (ip=100.105.27.67, pid=42619) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=62733) - (ip=100.105.27.67, pid=42620) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=62733) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=42619, ip=100.105.27.67) Moving model to device: cpu
(RayTrainWorker pid=42619, ip=100.105.27.67) Wrapping provided model in DistributedDataParallel.


(pid=63023) Running Dataset train_10_0.: 0.00 row [00:00, ? row/s]

(pid=63023) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=63023) Running Dataset train_10_4.: 0.00 row [00:00, ? row/s]

(pid=63023) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=63023) Registered dataset logger for dataset train_10_4
(SplitCoordinator pid=63023) Starting execution of Dataset train_10_4. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data
(SplitCoordinator pid=63023) Execution plan of Dataset train_10_4: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=63023) ✔️  Dataset train_10_4 execution finished in 0.02 seconds
(TrainController pid=62733) [State Transition] RUNNING -> SHUTTING_DOWN.
(RayTrainWorker pid=42620, ip=100.105.27.67) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_09-44-53/checkpoint_2026-08-24_09-45-14.815667) [repeated 9x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(RayTrainWorker pid=42620, ip=

,loss,epoch
0,1.595301,0
1,0.804270,1
2,0.158404,2
3,0.017133,3
4,0.003269,4


(TrainController pid=62733) [State Transition] SHUTTING_DOWN -> FINISHED.


## Optimize hyperparameters with Ray Tune

Tune allows us to maximize cluster utilization by running multiple experiments, each of which may require multiple workers/GPUs.

The example below features simple random search, buy by using built-in integrations for efficient schedulers and search algorithms, we can immediately use knowledge from completed trials to schedule additional trials.

In [7]:
tuner = Tuner(
    tune.with_parameters(train_driver, dataset=train_ds),
    param_space={        
        "lr": tune.loguniform(1e-4, 1e-1), # example: value Tune actually searches over
    },
    tune_config=TuneConfig(
        metric="loss",
        mode="min",
        num_samples=3,        
    ),
)

tune_results = tuner.fit()

(TrainController pid=43695, ip=100.71.58.72) Requesting resources: {'CPU': 1} * 2
(TrainController pid=43695, ip=100.71.58.72) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=43695, ip=100.71.58.72) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(RayTrainWorker pid=43809, ip=100.71.58.72) Setting up process group for: env:// [rank=0, world_size=2]


(RayTrainWorker pid=43809, ip=100.71.58.72) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(TrainController pid=45364, ip=100.105.27.67) Requesting resources: {'CPU': 1} * 2 [repeated 2x across cluster]
(TrainController pid=45364, ip=100.105.27.67) [State Transition] INITIALIZING -> SCHEDULING. [repeated 2x across cluster]
(TrainController pid=45364, ip=100.105.27.67) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2 [repeated 2x across cluster]
(TrainController pid=43695, ip=100.71.58.72) Started training worker group of size 2: 
(TrainController pid=43695, ip=100.71.58.72) - (ip=100.71.58.72, pid=43809) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=43695, ip=100.71.58.72) - (ip=100.71.58.72, pid=43808) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=43695, ip=100.71.58.72) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=43809, ip=100.71.58.72) Moving model to device: cpu
(RayTrainWorker pid=43809, ip=100.71.58.72) Wrapping provided model in DistributedDataParallel.
(SplitCoordinator 

(pid=45945, ip=100.105.27.67) Running Dataset train_12_0.: 0.00 row [00:00, ? row/s]

(pid=45945, ip=100.105.27.67) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

...

(pid=44148, ip=100.71.58.72) Running Dataset train_16_4.: 0.00 row [00:00, ? row/s]

(pid=44148, ip=100.71.58.72) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=46072, ip=100.105.27.67) Running Dataset train_14_4.: 0.00 row [00:00, ? row/s]

(pid=46072, ip=100.105.27.67) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=46072, ip=100.105.27.67) Registered dataset logger for dataset train_14_4 [repeated 7x across cluster]
(TrainController pid=45364, ip=100.105.27.67) [State Transition] RUNNING -> SHUTTING_DOWN.
(SplitCoordinator pid=46072, ip=100.105.27.67) Starting execution of Dataset train_14_4. Full logs are in /tmp/ray/session_2026-08-24_05-38-11_388307_2657/logs/ray-data [repeated 7x across cluster]
(SplitCoordinator pid=46072, ip=100.105.27.67) Execution plan of Dataset train_14_4: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)] [repeated 7x across cluster]
(SplitCoordinator pid=46072, ip=100.105.27.67) ✔️  Dataset train_14_4 execution finished in 0.02 seconds [repeated 7x across cluster]
(RayTrainWorker pid=43829, ip=100.71.58.72) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_09-58-45/checkpoint_2026-08-24_09-59-07.824979) [repeated 9x across cluster]
(RayTrainWorker pid=43829, ip=100.71.5

In [8]:
tune_results.get_dataframe()

,loss,epoch,checkpoint_path,timestamp,checkpoint_dir_name,done,training_iteration,trial_id,date,time_this_iter_s,time_total_s,pid,hostname,node_ip,time_since_restore,iterations_since_restore,config/lr,logdir
0,0.171733,4,/mnt/cluster_storage/ray_train_run-2026-08-24_...,1787590751,None,False,5,0e183_00000,2026-08-24_09-59-11,2.074405,25.698368,43630,ip-10-0-19-50,100.71.58.72,25.698368,5,0.004774,0e183_00000
1,1.170063,4,/mnt/cluster_storage/ray_train_run-2026-08-24_...,1787590751,None,False,5,0e183_00001,2026-08-24_09-59-11,2.085623,26.288277,45252,ip-10-0-14-61,100.105.27.67,26.288277,5,0.001902,0e183_00001
2,0.003205,4,/mnt/cluster_storage/ray_train_run-2026-08-24_...,1787590749,None,False,5,0e183_00002,2026-08-24_09-59-09,2.136093,23.863031,45251,ip-10-0-14-61,100.105.27.67,23.863031,5,0.009998,0e183_00002


In [9]:
best_result_path = tune_results.get_best_result("loss", mode="min").metrics['checkpoint_path']

best_result_path

'/mnt/cluster_storage/ray_train_run-2026-08-24_09-58-45/checkpoint_2026-08-24_09-59-05.314187'

(TrainController pid=45363, ip=100.105.27.67) [State Transition] SHUTTING_DOWN -> FINISHED.


## Batch inference with Ray Data

Batch inference is implemented similar to feature engineering using stateful computation. In the case of inference, the state is the model we want to load and re-use for many batches of data.

In [10]:
class OfflinePredictor:
    def __init__(self):
        # Load expensive state
        self._model = ProductClassifier()
        self._model.load_state_dict(torch.load(best_result_path + '/model.pt', weights_only=True))

    def __call__(self, batch: dict) -> dict:
        # Make prediction in batch
        with torch.inference_mode():
            outputs = self._model(torch.tensor(batch['features']))
        return {"prediction": outputs.numpy()}

In [ ]:
predictions = val_ds.select_columns(['features']).map_batches(OfflinePredictor, concurrency=2)
predictions.take_batch(3)

## Online prediction with Ray Serve

For low-latency, high-performance serving, we code and enhance a simple Python class to create a `Deployment`. Deployments can be composed to allow easy integration while retaining good separation-of-concerns patterns, effective development, and simple upgrades.

In [ ]:
@serve.deployment
class OnlinePredictor:
    def __init__(self, checkpoint):
        self._model = ProductClassifier()
        self._model.load_state_dict(torch.load(checkpoint, weights_only=True))

    async def __call__(self, request: Request) -> dict:
        data = await request.json()
        return {"prediction": self.predict(data)}

    def predict(self, data: dict) -> list[float]:
        with torch.inference_mode():
            outputs = self._model(torch.tensor(data["features"]))
        return outputs.numpy().tolist()

handle = serve.run(OnlinePredictor.bind(checkpoint=best_result_path + '/model.pt'))

In [ ]:
# Form payload
sample_inputs = val_ds.select_columns(['features'])
sample = sample_inputs.take(1)[0]['features'].astype('float64')

# Send HTTP request
requests.post("http://localhost:8000/", json={'features' : list(sample)}).json()

In [ ]:
# Shutdown Ray Serve
serve.shutdown()

# A Brief Look at Ray Core

## Ray Core overview

Ray Core is about:
* distributing computation across many cores, nodes, or devices (e.g., accelerators)
* scheduling *arbitrary task graphs*
    * any code you can write, you can distribute, scale, and accelerate with Ray Core
* manage the overhead
    * at scale, distributed computation introduces growing "frictions" -- data movement, scheduling costs, etc. -- which make the problem harder
    * Ray Core addresses these issues as first-order concerns in its design (e.g., via a distributed scheduler)
 
(And, of course, for common technical use cases, libraries and other components provide simple dev ex and are built on top of Ray Core)

## `@ray.remote` and `ray.get`

Here is a diagram which shows the relationship between Python code and Ray tasks.

<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Ray_Core/python_to_ray_task_map.png" width="80%" >

Define a Python function and decorate it so that Ray can schedule it

In [ ]:
@ray.remote(num_cpus=2)
def f(a, b):
    return a + b

Tell Ray to schedule the function

In [ ]:
f.remote(1, 2)

`ObjectRef` is a handle to a task result. We get an ObjectRef immediately because we don't know
* when the task will run
* whether it will succeed
* whether we really need or want the result locally
    * consider a very large result which we may need for other work but which we don't need to inspect

In [ ]:
ref = f.remote(1, 2)

If we want to wait (block) and retrieve the corresponding object, we can use `ray.get`

In [ ]:
ray.get(ref)

### Task graphs

The above example is a common scenario, but it is also the easiest (least complex) scheduling scenario. Each task is independent of the others -- this is called "embarrassingly parallel"

Many real-world algorithms are not embarrassingly parallel: some tasks depend on results from one or more other tasks. Scheduling this graphs is more challenging.

Ray Core is designed to make this straightforward

In [ ]:
@ray.remote
def add(a, b):
    return a+b

In [ ]:
arg1 = add.remote(1,2)

arg1

In [ ]:
arg2 = add.remote(10, 20)

We want to schedule `add` which depends on two prior invocations of `add`

We can pass the resulting ObjectRefs -- this means 
* we don't have to wait for the dependencies to complete before we can set up `add` for scheduling
* we don't need to have the concrete parameters (Python objects) for the call to `add.remote`
* Ray will automatically resolve the ObjectRefs -- our `add` implementation will never know that we passed ObjectRefs, not, e.g., numbers

In [ ]:
out = add.remote(arg1, arg2)

In [ ]:
ray.get(out)

## Ray Actors

Actors are Python class instances which can run for a long time in the cluster, which can maintain state, and which can send messages to/from other code.

Let's look at an example of an actor which maintains a running balance.

In [ ]:
@ray.remote
class Accounting:
    def __init__(self):
        self._total = 0
    
    def add(self, amount):
        self._total += amount
        
    def remove(self, amount):
        self._total -= amount
        
    def total(self):
        return self._total

<div class="alert alert-block alert-warning">

<b>Note:</b> The most common use case for actors is with state that is not mutated but is large enough that we may want to load it only once and ensure we can route calls to it over time, such as a large AI model.

</div>

Define an actor with the `@ray.remote` decorator and then use `<class_name>.remote()` ask Ray to construct and instance of this actor somewhere in the cluster.

We get an actor handle which we can use to communicate with that actor, pass to other code, tasks, or actors, etc.

In [ ]:
acc = Accounting.remote()

We can send a message to an actor -- with RPC semantics -- by using `<handle>.<method_name>.remote()`

In [ ]:
acc.total.remote()

Not surprisingly, we get an object ref back

In [ ]:
ray.get(acc.total.remote())

We can mutate the state inside this actor instance

In [ ]:
acc.add.remote(100)

In [ ]:
acc.remove.remote(10)

In [ ]:
ray.get(acc.total.remote())